In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, CLIPProcessor, CLIPVisionModel
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import os
import gc

from sklearn.metrics import classification_report
# --- CONFIGURATION ---
class Config:
    # Paths
    CSV_PATH = "Tamil_Dataset_Training.xlsx"  # Your specific file
    IMG_DIR = "/content/images"              # Update this to your image folder path
    MODEL_SAVE_NAME = "tamil_only_model.pth"

    # Model Settings
    TEXT_MODEL = "google/muril-base-cased"
    VISION_MODEL = "openai/clip-vit-base-patch32"
    MAX_LEN = 128
    BATCH_SIZE = 8 # Reduced batch size to mitigate OOM
    EPOCHS = 5
    LR = 2e-5

    # Labels (Tamil Only)
    NUM_L1_CLASSES = 2  # Troll vs Support
    NUM_L2_CLASSES = 2  # Person vs Party (No Intersection in Tamil)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 1. DATASET CLASS ---
class TamilMemeDataset(Dataset):
    def __init__(self, df, tokenizer, processor, img_dir, max_len):
        self.df = df
        self.tokenizer = tokenizer
        self.processor = processor
        self.img_dir = img_dir
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # 1. Image Processing
        img_name = row['Image_name']
        img_path = os.path.join(self.img_dir, img_name)

        try:
            image = Image.open(img_path).convert("RGB")
        except:
            # Fallback for missing images: Black image
            image = Image.new('RGB', (224, 224), (0, 0, 0))

        # CLIP Processor (Resizes & Normalizes)
        pixel_values = self.processor(images=image, return_tensors="pt").pixel_values.squeeze(0)

        # 2. Text Processing (Using OCR column)
        text = str(row['ocr-text']) if pd.notna(row['ocr-text']) else ""

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'pixel_values': pixel_values,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'l1_label': torch.tensor(row['L1_Label'], dtype=torch.long),
            'l2_label': torch.tensor(row['L2_Label'], dtype=torch.long)
        }

# --- 2. THE MODEL (CLIP + MuRIL) ---
class TamilFusionModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoders
        self.text_encoder = AutoModel.from_pretrained(Config.TEXT_MODEL)
        self.vision_encoder = CLIPVisionModel.from_pretrained(Config.VISION_MODEL)

        # Dimensions
        self.text_dim = 768
        self.vis_dim = 768
        self.fusion_dim = 512

        # Fusion Layers
        self.project_t = nn.Linear(self.text_dim, self.fusion_dim)
        self.project_v = nn.Linear(self.vis_dim, self.fusion_dim)
        self.dropout = nn.Dropout(0.3)

        # Classification Heads
        # L1: Troll(0) vs Support(1)
        self.head_l1 = nn.Linear(self.fusion_dim * 2, Config.NUM_L1_CLASSES)

        # L2: Person(0) vs Party(1) -> Conditioned on L1
        # Input size = Fusion (1024) + L1 Logits (2)
        self.head_l2 = nn.Linear(self.fusion_dim * 2 + Config.NUM_L1_CLASSES, Config.NUM_L2_CLASSES)

    def forward(self, input_ids, attention_mask, pixel_values):
        # Text Features
        txt_out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        txt_emb = txt_out.pooler_output # [Batch, 768]

        # Image Features
        vis_out = self.vision_encoder(pixel_values=pixel_values)
        vis_emb = vis_out.pooler_output # [Batch, 768]

        # Projection
        t_proj = self.project_t(txt_emb)
        v_proj = self.project_v(vis_emb)

        # Fusion (Concatenation)
        fusion = torch.cat((t_proj, v_proj), dim=1) # [Batch, 1024]
        fusion = self.dropout(fusion)

        # L1 Prediction
        logits_l1 = self.head_l1(fusion)

        # L2 Prediction (Hierarchical)
        l2_input = torch.cat((fusion, logits_l1), dim=1)
        logits_l2 = self.head_l2(l2_input)

        return logits_l1, logits_l2

# --- 3. TRAINING LOOP ---
def train_tamil():
    # Force garbage collection to clear memory from previous runs
    gc.collect()
    torch.cuda.empty_cache()

    print(f"Loading data from {Config.CSV_PATH}...")
    df = pd.read_excel(Config.CSV_PATH)

    # Split Train/Val
    train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)
    print(f"Train Size: {len(train_df)} | Val Size: {len(val_df)}")

    # Tokenizers
    tokenizer = AutoTokenizer.from_pretrained(Config.TEXT_MODEL)
    processor = CLIPProcessor.from_pretrained(Config.VISION_MODEL)

    # Datasets
    train_ds = TamilMemeDataset(train_df, tokenizer, processor, Config.IMG_DIR, Config.MAX_LEN)
    val_ds = TamilMemeDataset(val_df, tokenizer, processor, Config.IMG_DIR, Config.MAX_LEN)

    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE)

    # Initialize Model
    model = TamilFusionModel().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LR)

    # Loss Weights (Handling Imbalance)
    # L1: Support is rare (~15%), weigh it higher
    weights_l1 = torch.tensor([1.0, 2.5]).to(device)
    loss_fn_l1 = nn.CrossEntropyLoss(weight=weights_l1)

    # L2: Person is dominant, weigh Party higher
    weights_l2 = torch.tensor([1.0, 2.0]).to(device)
    loss_fn_l2 = nn.CrossEntropyLoss(weight=weights_l2)

    print("🚀 Starting Training (Tamil Only)...")

    for epoch in range(Config.EPOCHS):
        model.train()
        total_loss = 0

        for batch in tqdm(train_loader):
            # Move to GPU
            input_ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            pixels = batch['pixel_values'].to(device)
            l1_target = batch['l1_label'].to(device)
            l2_target = batch['l2_label'].to(device)

            optimizer.zero_grad()

            # Forward
            l1_logits, l2_logits = model(input_ids, mask, pixels)

            # Multi-Task Loss
            loss1 = loss_fn_l1(l1_logits, l1_target)
            loss2 = loss_fn_l2(l2_logits, l2_target)
            loss = loss1 + loss2

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{Config.EPOCHS} | Loss: {avg_loss:.4f}")

        # Validation
        model.eval()
        # correct_l1 = 0
        # correct_l2 = 0
        # total = 0

        all_l1_preds = []
        all_l1_labels = []
        all_l2_preds = []
        all_l2_labels = []

        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                mask = batch['attention_mask'].to(device)
                pixels = batch['pixel_values'].to(device)
                l1_target = batch['l1_label'].to(device)
                l2_target = batch['l2_label'].to(device)

                l1_out, l2_out = model(input_ids, mask, pixels)

                pred_l1 = torch.argmax(l1_out, dim=1)
                pred_l2 = torch.argmax(l2_out, dim=1)

                # correct_l1 += (pred_l1 == l1_target).sum().item()
                # correct_l2 += (pred_l2 == l2_target).sum().item()
                # total += l1_target.size(0)

                all_l1_preds.extend(pred_l1.cpu().numpy())
                all_l1_labels.extend(l1_target.cpu().numpy())

                all_l2_preds.extend(pred_l2.cpu().numpy())
                all_l2_labels.extend(l2_target.cpu().numpy())

        # print(f"Val L1 Acc: {correct_l1/total:.4f} | Val L2 Acc: {correct_l2/total:.4f}")

        print("\n Validation Classification Report \u2014 L1 (Troll vs Support)")
        print(
          classification_report(
            all_l1_labels,
            all_l1_preds,
            target_names=["Troll", "Support"],
            digits=4
          )
        )

        print("\n Validation Classification Report \u2014 L2 (Person vs Party)")
        print(
          classification_report(
            all_l2_labels,
            all_l2_preds,
            target_names=["Person", "Party"],
            digits=4
        )
      )


    # Save
    torch.save(model.state_dict(), Config.MODEL_SAVE_NAME)
    print(f"\u2705 Model Saved: {Config.MODEL_SAVE_NAME}")

if __name__ == "__main__":
    train_tamil()

Loading data from Tamil_Dataset_Training.xlsx...
Train Size: 722 | Val Size: 81


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias 

...

Epoch 3/5 | Loss: 0.7679

 Validation Classification Report — L1 (Troll vs Support)
              precision    recall  f1-score   support

       Troll     0.9718    0.9452    0.9583        73
     Support     0.6000    0.7500    0.6667         8

    accuracy                         0.9259        81
   macro avg     0.7859    0.8476    0.8125        81
weighted avg     0.9351    0.9259    0.9295        81


 Validation Classification Report — L2 (Person vs Party)
              precision    recall  f1-score   support

      Person     0.8732    0.8986    0.8857        69
       Party     0.3000    0.2500    0.2727        12

    accuracy                         0.8025        81
   macro avg     0.5866    0.5743    0.5792        81
weighted avg     0.7883    0.8025    0.7949        81



100%|██████████| 91/91 [00:39<00:00,  2.32it/s]


Epoch 4/5 | Loss: 0.4983

 Validation Classification Report — L1 (Troll vs Support)
              precision    recall  f1-score   support

       Troll     0.9125    1.0000    0.9542        73
     Support     1.0000    0.1250    0.2222         8

    accuracy                         0.9136        81
   macro avg     0.9563    0.5625    0.5882        81
weighted avg     0.9211    0.9136    0.8819        81


 Validation Classification Report — L2 (Person vs Party)
              precision    recall  f1-score   support

      Person     0.8529    0.8406    0.8467        69
       Party     0.1538    0.1667    0.1600        12

    accuracy                         0.7407        81
   macro avg     0.5034    0.5036    0.5034        81
weighted avg     0.7494    0.7407    0.7450        81



100%|██████████| 91/91 [00:39<00:00,  2.31it/s]


Epoch 5/5 | Loss: 0.3423

 Validation Classification Report — L1 (Troll vs Support)
              precision    recall  f1-score   support

       Troll     0.9231    0.9863    0.9536        73
     Support     0.6667    0.2500    0.3636         8

    accuracy                         0.9136        81
   macro avg     0.7949    0.6182    0.6586        81
weighted avg     0.8978    0.9136    0.8954        81


 Validation Classification Report — L2 (Person vs Party)
              precision    recall  f1-score   support

      Person     0.8571    0.9565    0.9041        69
       Party     0.2500    0.0833    0.1250        12

    accuracy                         0.8272        81
   macro avg     0.5536    0.5199    0.5146        81
weighted avg     0.7672    0.8272    0.7887        81

✅ Model Saved: tamil_only_model.pth


# Test


In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, CLIPProcessor, AutoModel, CLIPVisionModel
from PIL import Image
from tqdm import tqdm
import os

# --- CONFIG ---
TEST_DIR = "/content/test_tamil"  # Update this to your Test Images folder
MODEL_PATH = "tamil_only_model.pth"
OUTPUT_CSV = "submission_tamil.csv"

# Model Config (Must match training!)
TEXT_MODEL = "google/muril-base-cased"
VISION_MODEL = "openai/clip-vit-base-patch32"
MAX_LEN = 128
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 1. MODEL ARCHITECTURE (Must be identical to training) ---
class TamilFusionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.text_encoder = AutoModel.from_pretrained(TEXT_MODEL)
        self.vision_encoder = CLIPVisionModel.from_pretrained(VISION_MODEL)
        self.text_dim = 768
        self.vis_dim = 768
        self.fusion_dim = 512

        self.project_t = nn.Linear(self.text_dim, self.fusion_dim)
        self.project_v = nn.Linear(self.vis_dim, self.fusion_dim)
        self.dropout = nn.Dropout(0.3)

        self.head_l1 = nn.Linear(self.fusion_dim * 2, 2)
        self.head_l2 = nn.Linear(self.fusion_dim * 2 + 2, 2)

    def forward(self, input_ids, attention_mask, pixel_values):
        txt_out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        txt_emb = txt_out.pooler_output
        vis_out = self.vision_encoder(pixel_values=pixel_values)
        vis_emb = vis_out.pooler_output

        t_proj = self.project_t(txt_emb)
        v_proj = self.project_v(vis_emb)
        fusion = torch.cat((t_proj, v_proj), dim=1)
        fusion = self.dropout(fusion)

        logits_l1 = self.head_l1(fusion)
        l2_input = torch.cat((fusion, logits_l1), dim=1)
        logits_l2 = self.head_l2(l2_input)

        return logits_l1, logits_l2

# --- 2. TEST DATASET ---
class TestDataset(Dataset):
    def __init__(self, img_dir, tokenizer, processor, max_len):
        self.img_dir = img_dir
        # Only take valid images
        self.images = [f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
        self.tokenizer = tokenizer
        self.processor = processor
        self.max_len = max_len

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)

        try:
            image = Image.open(img_path).convert("RGB")
        except:
            image = Image.new('RGB', (224, 224), (0, 0, 0))

        pixel_values = self.processor(images=image, return_tensors="pt").pixel_values.squeeze(0)

        # We don't have OCR for test set yet, so input empty string
        # (Or run OCR on test set if you want higher accuracy)
        text = ""

        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'image_name': img_name,
            'pixel_values': pixel_values,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }

# --- 3. DECODER FUNCTION (The Critical Mapping) ---
def decode_labels(l1_id, l2_id):
    # L1: Sentiment
    l1_text = "Support/Praise" if l1_id == 1 else "Troll/Oppose"

    # L2: Target
    # Logic: 0=Person, 1=Party.
    # We must format it exactly: "Troll/Oppose Against X" or "Support for X"

    target = "Person" if l2_id == 0 else "Party"

    if l1_id == 1: # Support
        # Note: 'person' is lowercase in 'Support for person' in your train data
        target_str = "person" if l2_id == 0 else "party"
        l2_text = f"Support for {target_str}"
    else: # Troll
        # Note: 'Person' is capitalized in 'Troll/Oppose Against Person'
        l2_text = f"Troll/Oppose Against {target}"

    return l1_text, l2_text

# --- 4. EXECUTION ---
def generate_submission():
    print("🚀 Loading Model...")
    model = TamilFusionModel().to(device)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model.eval()

    tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL)
    processor = CLIPProcessor.from_pretrained(VISION_MODEL)

    print(f"📂 Reading Test Images from {TEST_DIR}...")
    ds = TestDataset(TEST_DIR, tokenizer, processor, MAX_LEN)
    loader = DataLoader(ds, batch_size=32, shuffle=False)

    results = []

    with torch.no_grad():
        for batch in tqdm(loader):
            input_ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            pixels = batch['pixel_values'].to(device)
            img_names = batch['image_name']

            # Predict
            l1_logits, l2_logits = model(input_ids, mask, pixels)

            l1_preds = torch.argmax(l1_logits, dim=1).cpu().numpy()
            l2_preds = torch.argmax(l2_logits, dim=1).cpu().numpy()

            # Decode
            for i, name in enumerate(img_names):
                l1_txt, l2_txt = decode_labels(l1_preds[i], l2_preds[i])
                results.append({
                    "Image_name": name,
                    "Level1": l1_txt,
                    "Level2": l2_txt
                })

    # Save
    df = pd.DataFrame(results)
    # Sort nicely if needed
    # df = df.sort_values('Image_name')

    df.to_csv(OUTPUT_CSV, index=False)
    print(f"✅ Submission Saved: {OUTPUT_CSV}")
    print(df.head())

if __name__ == "__main__":
    generate_submission()

🚀 Loading Model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias 

📂 Reading Test Images from /content/test_tamil...


100%|██████████| 7/7 [00:03<00:00,  1.81it/s]


✅ Submission Saved: submission_tamil.csv
  Image_name        Level1                       Level2
0    603.jpg  Troll/Oppose  Troll/Oppose Against Person
1    512.jpg  Troll/Oppose  Troll/Oppose Against Person
2    441.jpg  Troll/Oppose  Troll/Oppose Against Person
3    006.jpg  Troll/Oppose  Troll/Oppose Against Person
4    314.jpg  Troll/Oppose  Troll/Oppose Against Person
